In [1]:
# import kagglehub
# # let's authintificate ourselves with API token
# kagglehub.login()

In [2]:
# playground_series_s5e1_path = kagglehub.competition_download('playground-series-s5e1')
# playground_series_s5e1_path

In [3]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('./.'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

././train.csv
././submission.csv
././test.csv
././playground-s5e1.ipynb
././playground-series-s5e1-mess.ipynb
././catboost_info/catboost_training.json
././catboost_info/time_left.tsv
././catboost_info/learn_error.tsv
././catboost_info/learn/events.out.tfevents


In [4]:
import sklearn
print(sklearn.__version__)

1.6.0


In [5]:
df_train = pd.read_csv('train.csv')
df_train

,id,date,country,store,product,num_sold
0,0,2010-01-01,Canada,Discount Stickers,Holographic Goose,NaN
1,1,2010-01-01,Canada,Discount Stickers,Kaggle,973.0
2,2,2010-01-01,Canada,Discount Stickers,Kaggle Tiers,906.0
3,3,2010-01-01,Canada,Discount Stickers,Kerneler,423.0
4,4,2010-01-01,Canada,Discount Stickers,Kerneler Dark Mode,491.0
...,...,...,...,...,...,...
230125,230125,2016-12-31,Singapore,Premium Sticker Mart,Holographic Goose,466.0
230126,230126,2016-12-31,Singapore,Premium Sticker Mart,Kaggle,2907.0
230127,230127,2016-12-31,Singapore,Premium Sticker Mart,Kaggle Tiers,2299.0
230128,230128,2016-12-31,Singapore,Premium Sticker Mart,Kerneler,1242.0


In [6]:
df_train.drop('id', axis=1, inplace=True)

In [7]:
print(df_train.count())

date        230130
country     230130
store       230130
product     230130
num_sold    221259
dtype: int64


In [8]:
# df_train['year-month'] = df_train['date'].apply(lambda x: x[:7])
# df = df_train[(df_train['country'] == 'Canada') & (df_train['store'] == 'Discount Stickers') & (df_train['product'] == 'Holographic Goose')]
# df = df[['year-month', 'num_sold']]
# df_agg = df.groupby(['year-month']).sum()
# df_agg

In [9]:
# import seaborn as sns
# import matplotlib.pyplot as plt

# for country in df_train.country.unique():
#     for store in df_train.store.unique():
#         for product in df_train['product'].unique():
#             plt.figure(figsize=(10, 6))
#             plt.title(f'Number of Sold Products by Year-Month for {country}, {store}, and {product}')
#             df = df_train[(df_train['country'] == country) & (df_train['store'] == store) & (df_train['product'] == product)]
#             df = df[['year-month', 'num_sold']]
#             df_agg = df.groupby(['year-month']).sum()
#             sns.lineplot(x='year-month', y='num_sold', data=df_agg)
#             plt.show()


In [10]:
df_train['year'] = df_train['date'].apply(lambda x: int(x[:4]))
df_train['month'] = df_train['date'].apply(lambda x: int(x[5:7]))
df_train['day'] = df_train['date'].apply(lambda x: int(x[8:10]))
df_train['date'] = pd.to_datetime(df_train['date'])
df_train['day of year'] = df_train['date'].dt.dayofyear
df_train['day of week'] = df_train['date'].dt.weekday
df_train['sin day of week'] = np.sin(2 * np.pi * df_train['day of week'] / 7)
df_train['cos day of week'] = np.cos(2 * np.pi * df_train['day of week'] / 7)
df_train['sin day of month'] = np.sin(2 * np.pi * df_train['day'] / 30)
df_train['cos day of month'] = np.cos(2 * np.pi * df_train['day'] / 30)
df_train['sin of month'] = np.sin(2 * np.pi * df_train['month'] / 12)
df_train['cos of month'] = np.cos(2 * np.pi * df_train['month'] / 12)
#df_train['seconds since 1970'] = df_train['date'].astype('int64') // 10 ** 9
df_train.drop('date', axis=1, inplace=True) #
df_train

,country,store,product,num_sold,year,month,day,day of year,day of week,sin day of week,cos day of week,sin day of month,cos day of month,sin of month,cos of month
0,Canada,Discount Stickers,Holographic Goose,NaN,2010,1,1,1,4,-0.433884,-0.900969,0.207912,0.978148,5.000000e-01,0.866025
1,Canada,Discount Stickers,Kaggle,973.0,2010,1,1,1,4,-0.433884,-0.900969,0.207912,0.978148,5.000000e-01,0.866025
2,Canada,Discount Stickers,Kaggle Tiers,906.0,2010,1,1,1,4,-0.433884,-0.900969,0.207912,0.978148,5.000000e-01,0.866025
3,Canada,Discount Stickers,Kerneler,423.0,2010,1,1,1,4,-0.433884,-0.900969,0.207912,0.978148,5.000000e-01,0.866025
4,Canada,Discount Stickers,Kerneler Dark Mode,491.0,2010,1,1,1,4,-0.433884,-0.900969,0.207912,0.978148,5.000000e-01,0.866025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
230125,Singapore,Premium Sticker Mart,Holographic Goose,466.0,2016,12,31,366,5,-0.974928,-0.222521,0.207912,0.978148,-2.449294e-16,1.000000
230126,Singapore,Premium Sticker Mart,Kaggle,2907.0,2016,12,31,366,5,-0.974928,-0.222521,0.207912,0.978148,-2.449294e-16,1.000000
230127,Singapore,Premium Sticker Mart,Kaggle Tiers,2299.0,2016,12,31,366,5,-0.974928,-0.222521,0.207912,0.978148,-2.449294e-16,1.000000
230128,Singapore,Premium Sticker Mart,Kerneler,1242.0,2016,12,31,366,5,-0.974928,-0.222521,0.207912,0.978148,-2.449294e-16,1.000000


In [11]:
def season(month):
    match month:
        case 1 | 12: 
            return 'winter'
        case 2 | 3 | 4 | 5:
            return 'spring'
        case 6 | 7 | 8 :
            return 'summer'
        case _:
            return 'autumn'

df_train['season'] = df_train['month'].apply(season) 

In [12]:
df_train['sin day of year'] = np.sin(2 * np.pi * df_train['day of year'] / 365)
df_train['cos day of year'] = np.cos(2 * np.pi * df_train['day of year'] / 365)


df_train['day'] = df_train['day'].astype('category')
df_train['month'] = df_train['month'].astype('category')


df_train['day of year'] = df_train['day of year'].astype('category')
df_train['day of week'] = df_train['day of week'].astype('category')


df_train

,country,store,product,num_sold,year,month,day,day of year,day of week,sin day of week,cos day of week,sin day of month,cos day of month,sin of month,cos of month,season,sin day of year,cos day of year
0,Canada,Discount Stickers,Holographic Goose,NaN,2010,1,1,1,4,-0.433884,-0.900969,0.207912,0.978148,5.000000e-01,0.866025,winter,0.017213,0.999852
1,Canada,Discount Stickers,Kaggle,973.0,2010,1,1,1,4,-0.433884,-0.900969,0.207912,0.978148,5.000000e-01,0.866025,winter,0.017213,0.999852
2,Canada,Discount Stickers,Kaggle Tiers,906.0,2010,1,1,1,4,-0.433884,-0.900969,0.207912,0.978148,5.000000e-01,0.866025,winter,0.017213,0.999852
3,Canada,Discount Stickers,Kerneler,423.0,2010,1,1,1,4,-0.433884,-0.900969,0.207912,0.978148,5.000000e-01,0.866025,winter,0.017213,0.999852
4,Canada,Discount Stickers,Kerneler Dark Mode,491.0,2010,1,1,1,4,-0.433884,-0.900969,0.207912,0.978148,5.000000e-01,0.866025,winter,0.017213,0.999852
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
230125,Singapore,Premium Sticker Mart,Holographic Goose,466.0,2016,12,31,366,5,-0.974928,-0.222521,0.207912,0.978148,-2.449294e-16,1.000000,winter,0.017213,0.999852
230126,Singapore,Premium Sticker Mart,Kaggle,2907.0,2016,12,31,366,5,-0.974928,-0.222521,0.207912,0.978148,-2.449294e-16,1.000000,winter,0.017213,0.999852
230127,Singapore,Premium Sticker Mart,Kaggle Tiers,2299.0,2016,12,31,366,5,-0.974928,-0.222521,0.207912,0.978148,-2.449294e-16,1.000000,winter,0.017213,0.999852
230128,Singapore,Premium Sticker Mart,Kerneler,1242.0,2016,12,31,366,5,-0.974928,-0.222521,0.207912,0.978148,-2.449294e-16,1.000000,winter,0.017213,0.999852


In [13]:
df_test = df_train[df_train['num_sold'].isna()]

In [14]:
# let's drop all rows with NaN values in the target column
df_train.dropna(subset=['num_sold'], inplace=True)

In [15]:
df_test.drop('num_sold', axis=1, inplace=True)
df_test

/tmp/ipykernel_50271/317333425.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test.drop('num_sold', axis=1, inplace=True)


,country,store,product,year,month,day,day of year,day of week,sin day of week,cos day of week,sin day of month,cos day of month,sin of month,cos of month,season,sin day of year,cos day of year
0,Canada,Discount Stickers,Holographic Goose,2010,1,1,1,4,-0.433884,-0.900969,2.079117e-01,0.978148,5.000000e-01,0.866025,winter,1.721336e-02,0.999852
45,Kenya,Discount Stickers,Holographic Goose,2010,1,1,1,4,-0.433884,-0.900969,2.079117e-01,0.978148,5.000000e-01,0.866025,winter,1.721336e-02,0.999852
90,Canada,Discount Stickers,Holographic Goose,2010,1,2,2,5,-0.974928,-0.222521,4.067366e-01,0.913545,5.000000e-01,0.866025,winter,3.442161e-02,0.999407
135,Kenya,Discount Stickers,Holographic Goose,2010,1,2,2,5,-0.974928,-0.222521,4.067366e-01,0.913545,5.000000e-01,0.866025,winter,3.442161e-02,0.999407
180,Canada,Discount Stickers,Holographic Goose,2010,1,3,3,6,-0.781831,0.623490,5.877853e-01,0.809017,5.000000e-01,0.866025,winter,5.161967e-02,0.998667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
229905,Kenya,Discount Stickers,Holographic Goose,2016,12,29,364,3,0.433884,-0.900969,-2.079117e-01,0.978148,-2.449294e-16,1.000000,winter,-1.721336e-02,0.999852
229950,Canada,Discount Stickers,Holographic Goose,2016,12,30,365,4,-0.433884,-0.900969,-1.133108e-15,1.000000,-2.449294e-16,1.000000,winter,6.432491e-16,1.000000
229995,Kenya,Discount Stickers,Holographic Goose,2016,12,30,365,4,-0.433884,-0.900969,-1.133108e-15,1.000000,-2.449294e-16,1.000000,winter,6.432491e-16,1.000000
230040,Canada,Discount Stickers,Holographic Goose,2016,12,31,366,5,-0.974928,-0.222521,2.079117e-01,0.978148,-2.449294e-16,1.000000,winter,1.721336e-02,0.999852


In [16]:
target = pd.DataFrame(df_train['num_sold'].astype('float64'))
target

,num_sold
1,973.0
2,906.0
3,423.0
4,491.0
5,300.0
...,...
230125,466.0
230126,2907.0
230127,2299.0
230128,1242.0


In [17]:
df_train.drop('num_sold', axis=1, inplace=True)

In [18]:
# from catboost import CatBoostRegressor, Pool

# train_pool = Pool(df_train, target, cat_features=['country', 'store', 'product'])

# regressor = CatBoostRegressor(iterations=300, loss_function='MAPE', verbose=True)

In [19]:
#regressor.fit(train_pool)

In [20]:
# ids = df_test['id']
# df_test.drop('id', axis=1, inplace=True)

In [21]:
# df_test['year'] = df_test['date'].apply(lambda x: int(x[:4]))
# df_test['month'] = df_test['date'].apply(lambda x: int(x[5:7]))
# df_test['day'] = df_test['date'].apply(lambda x: int(x[8:10]))
# df_test['date'] = pd.to_datetime(df_test['date'])
# df_test['day of year'] = df_test['date'].dt.dayofyear
# df_test['day of week'] = df_test['date'].dt.weekday
# df_test['sin day of week'] = np.sin(2 * np.pi * df_test['day of week'] / 7)
# df_test['cos day of week'] = np.cos(2 * np.pi * df_test['day of week'] / 7)
# df_test['sin day of month'] = np.sin(2 * np.pi * df_test['day'] / 30)
# df_test['cos day of month'] = np.cos(2 * np.pi * df_test['day'] / 30)
# df_test['sin of month'] = np.sin(2 * np.pi * df_test['month'] / 12)
# df_test['cos of month'] = np.cos(2 * np.pi * df_test['month'] / 12)
# #df_test['seconds since 1970'] = df_test['date'].astype('int64') // 10 ** 9
# df_test.drop('date', axis=1, inplace=True) #
# df_test

In [22]:
# pred = regressor.predict(df_test)

In [23]:
# df_test['season'] = df_test['month'].apply(season)

In [24]:
# df_test['day'] = df_test['day'].astype('category')
# df_test['month'] = df_test['month'].astype('category')

# df_test['sin day of year'] = np.sin(2 * np.pi * df_test['day of year'] / 365)
# df_test['cos day of year'] = np.cos(2 * np.pi * df_test['day of year'] / 365)

# df_test['day of year'] = df_test['day of year'].astype('category')
# df_test['day of week'] = df_test['day of week'].astype('category')


# df_test

In [25]:
from sklearn.pipeline import Pipeline

from sklearn.base import BaseEstimator, TransformerMixin

class DropNaNColumns(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.5):
        self.threshold = threshold
        self.columns_to_drop_ = None

    def fit(self, X, y=None):
        # Calculate the percentage of NaN values in each column
        nan_percentage = X.isna().mean()
        # Identify columns to drop
        self.columns_to_drop_ = nan_percentage[nan_percentage > self.threshold].index
        return self

    def transform(self, X):
        # Drop the identified columns
        return X.drop(columns=self.columns_to_drop_)


from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

def preprocess_data(X, threshold=0.5, n_neighbors=1500, encode=True, impute=True):
    # Separate numerical and categorical columns
    numerical_features = X.select_dtypes(include=['int64', 'float64']).columns
    categorical_features = X.select_dtypes(include=['object', 'category']).columns


    feature_names = X.columns
    # Numerical feature pipeline: Imputation + Scaling
    if impute:
        numerical_pipeline = Pipeline(steps=[
            ('dropnan', DropNaNColumns(threshold=threshold)),
            ('imputer', KNNImputer(n_neighbors=n_neighbors, weights='distance'))# SimpleImputer(strategy='mean')),#SimpleImputer()),#,
            # ('scaler', StandardScaler())  # Scale the numerical features
        ])
    else:
        numerical_pipeline = Pipeline(steps=[
            ('dropnan', DropNaNColumns(threshold=threshold)),
            #('imputer', KNNImputer(n_neighbors=n_neighbors, weights='distance'))# SimpleImputer(strategy='mean')),#SimpleImputer()),#,
            # ('scaler', StandardScaler())  # Scale the numerical features
        ])


    # Categorical feature pipeline: Imputation + Encoding
    if encode:
        categorical_pipeline = Pipeline(steps=[
            ('dropnan', DropNaNColumns(threshold=threshold)),
            # ('imputer', SimpleImputer(strategy='constant', fill_value='.')),#SimpleImputer(strategy='constant', fill_value='.')), 
            ('encoder', OneHotEncoder(handle_unknown='ignore')) 
        ])
    else:
        categorical_pipeline = Pipeline(steps=[
            ('dropnan', DropNaNColumns(threshold=threshold)),
            ('imputer', SimpleImputer(strategy='constant', fill_value='.')), 
        ])

    # Define transformers
    preprocessor = Pipeline(steps=[
        ('transformer', ColumnTransformer(
        transformers=[
            ('num', numerical_pipeline, numerical_features),
            ('cat', categorical_pipeline, categorical_features)
        ],
        verbose=True,
        n_jobs=-1
    )),
    ('scaler', StandardScaler(with_mean=False))
    ])

    return preprocessor

In [26]:
preprocessor = preprocess_data(df_train, threshold=0.8, encode=True, impute=False)
preprocessor

Pipeline(steps=[('transformer',
                 ColumnTransformer(n_jobs=-1,
                                   transformers=[('num',
                                                  Pipeline(steps=[('dropnan',
                                                                   DropNaNColumns(threshold=0.8))]),
                                                  Index(['year', 'sin day of week', 'cos day of week', 'sin day of month',
       'cos day of month', 'sin of month', 'cos of month', 'sin day of year',
       'cos day of year'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('dropnan',
                                                                   DropNaNColumns(threshold=0.8)),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['country', 'store', 'product', 'month', 'day', 'day of year',
       'day of week', 'season'],
      dtype='object'))],
                                   verbose=True)),
                ('scaler', StandardScaler(with_mean=False))])

In [27]:
from sklearn.base import BaseEstimator, RegressorMixin

class RegressorWrapper(RegressorMixin, BaseEstimator):
    def __init__(self, estimator):
        self.estimator = estimator

    def fit(self, X, y, eval_set=None, baseline=None, fobj=None):
        if baseline is not None or eval_set is not None:
          self.estimator.fit(X, y, eval_set=eval_set, baseline=baseline)
        if fobj is not None:
          self.estimator.fit(X, y, fobj=fobj)
        else:
          self.estimator.fit(X, y)
        return self

    def predict(self, X):
        return self.estimator.predict(X)


    @property
    def feature_importances_(self):
        if hasattr(self.estimator, 'feature_importances_'):
            return self.estimator.feature_importances_
        else:
            raise AttributeError("The underlying estimator does not have feature_importances_.")

    def __sklearn_is_fitted__(self):
        return hasattr(self.estimator, 'coef_') or hasattr(self.estimator, 'feature_importances_')

In [28]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import SGDRegressor

model_knn = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', KNeighborsRegressor(n_neighbors=35, weights='distance'))
])

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RegressorWrapper(LGBMRegressor(n_estimators=1000, objective='mape', boosting='dart', max_depth=7, bagging_freq=1, bagging_fraction=0.7, feature_fraction=0.7, n_jobs=-1))) # KNeighborsRegressor(n_neighbors=35, weights='distance'
    #)
    #RandomForestRegressor(n_estimators=500, criterion='absolute_error', max_depth=5, max_samples=0.8, max_features=0.7, verbose=1))
])

model

Pipeline(steps=[('preprocessor',
                 Pipeline(steps=[('transformer',
                                  ColumnTransformer(n_jobs=-1,
                                                    transformers=[('num',
                                                                   Pipeline(steps=[('dropnan',
                                                                                    DropNaNColumns(threshold=0.8))]),
                                                                   Index(['year', 'sin day of week', 'cos day of week', 'sin day of month',
       'cos day of month', 'sin of month', 'cos of month', 'sin day of year',
       'cos day of year'],
      dtype='object')),
                                                                  ('cat',
                                                                   Pipeline(steps=[('dro...
                                                                   Index(['country', 'store', 'product', 'month', 'day', 'day of year',
       'day of week', 'season'],
      dtype='object'))],
                                                    verbose=True)),
                                 ('scaler', StandardScaler(with_mean=False))])),
                ('regressor',
                 RegressorWrapper(estimator=LGBMRegressor(bagging_fraction=0.7,
                                                          bagging_freq=1,
                                                          boosting='dart',
                                                          feature_fraction=0.7,
                                                          max_depth=7,
                                                          n_estimators=1000,
                                                          n_jobs=-1,
                                                          objective='mape')))])

In [29]:
model_knn

Pipeline(steps=[('preprocessor',
                 Pipeline(steps=[('transformer',
                                  ColumnTransformer(n_jobs=-1,
                                                    transformers=[('num',
                                                                   Pipeline(steps=[('dropnan',
                                                                                    DropNaNColumns(threshold=0.8))]),
                                                                   Index(['year', 'sin day of week', 'cos day of week', 'sin day of month',
       'cos day of month', 'sin of month', 'cos of month', 'sin day of year',
       'cos day of year'],
      dtype='object')),
                                                                  ('cat',
                                                                   Pipeline(steps=[('dropnan',
                                                                                    DropNaNColumns(threshold=0.8)),
                                                                                   ('encoder',
                                                                                    OneHotEncoder(handle_unknown='ignore'))]),
                                                                   Index(['country', 'store', 'product', 'month', 'day', 'day of year',
       'day of week', 'season'],
      dtype='object'))],
                                                    verbose=True)),
                                 ('scaler', StandardScaler(with_mean=False))])),
                ('regressor',
                 KNeighborsRegressor(n_neighbors=35, weights='distance'))])

In [30]:
model_knn.fit(df_train, target)

[ColumnTransformer] ........... (1 of 2) Processing num, total=   0.0s
[ColumnTransformer] ........... (2 of 2) Processing cat, total=   0.2s


Pipeline(steps=[('preprocessor',
                 Pipeline(steps=[('transformer',
                                  ColumnTransformer(n_jobs=-1,
                                                    transformers=[('num',
                                                                   Pipeline(steps=[('dropnan',
                                                                                    DropNaNColumns(threshold=0.8))]),
                                                                   Index(['year', 'sin day of week', 'cos day of week', 'sin day of month',
       'cos day of month', 'sin of month', 'cos of month', 'sin day of year',
       'cos day of year'],
      dtype='object')),
                                                                  ('cat',
                                                                   Pipeline(steps=[('dropnan',
                                                                                    DropNaNColumns(threshold=0.8)),
                                                                                   ('encoder',
                                                                                    OneHotEncoder(handle_unknown='ignore'))]),
                                                                   Index(['country', 'store', 'product', 'month', 'day', 'day of year',
       'day of week', 'season'],
      dtype='object'))],
                                                    verbose=True)),
                                 ('scaler', StandardScaler(with_mean=False))])),
                ('regressor',
                 KNeighborsRegressor(n_neighbors=35, weights='distance'))])

In [31]:
pred = model_knn.predict(df_test)

In [32]:
df_train = pd.concat([df_train, df_test])
target = pd.concat([target, pd.DataFrame(pred, columns=['num_sold'])])
df_train

,country,store,product,year,month,day,day of year,day of week,sin day of week,cos day of week,sin day of month,cos day of month,sin of month,cos of month,season,sin day of year,cos day of year
1,Canada,Discount Stickers,Kaggle,2010,1,1,1,4,-0.433884,-0.900969,2.079117e-01,0.978148,5.000000e-01,0.866025,winter,1.721336e-02,0.999852
2,Canada,Discount Stickers,Kaggle Tiers,2010,1,1,1,4,-0.433884,-0.900969,2.079117e-01,0.978148,5.000000e-01,0.866025,winter,1.721336e-02,0.999852
3,Canada,Discount Stickers,Kerneler,2010,1,1,1,4,-0.433884,-0.900969,2.079117e-01,0.978148,5.000000e-01,0.866025,winter,1.721336e-02,0.999852
4,Canada,Discount Stickers,Kerneler Dark Mode,2010,1,1,1,4,-0.433884,-0.900969,2.079117e-01,0.978148,5.000000e-01,0.866025,winter,1.721336e-02,0.999852
5,Canada,Stickers for Less,Holographic Goose,2010,1,1,1,4,-0.433884,-0.900969,2.079117e-01,0.978148,5.000000e-01,0.866025,winter,1.721336e-02,0.999852
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
229905,Kenya,Discount Stickers,Holographic Goose,2016,12,29,364,3,0.433884,-0.900969,-2.079117e-01,0.978148,-2.449294e-16,1.000000,winter,-1.721336e-02,0.999852
229950,Canada,Discount Stickers,Holographic Goose,2016,12,30,365,4,-0.433884,-0.900969,-1.133108e-15,1.000000,-2.449294e-16,1.000000,winter,6.432491e-16,1.000000
229995,Kenya,Discount Stickers,Holographic Goose,2016,12,30,365,4,-0.433884,-0.900969,-1.133108e-15,1.000000,-2.449294e-16,1.000000,winter,6.432491e-16,1.000000
230040,Canada,Discount Stickers,Holographic Goose,2016,12,31,366,5,-0.974928,-0.222521,2.079117e-01,0.978148,-2.449294e-16,1.000000,winter,1.721336e-02,0.999852


In [33]:
target

,num_sold
1,973.000000
2,906.000000
3,423.000000
4,491.000000
5,300.000000
...,...
8866,263.461321
8867,668.880212
8868,279.588251
8869,649.493279


In [34]:
from sklearn.model_selection import train_test_split

X_train_first, X_train_second, target_first, target_second = train_test_split(df_train, target, test_size=0.2)

In [35]:
X_train_first

,country,store,product,year,month,day,day of year,day of week,sin day of week,cos day of week,sin day of month,cos day of month,sin of month,cos of month,season,sin day of year,cos day of year
49421,Canada,Premium Sticker Mart,Kaggle,2011,7,4,185,0,0.000000,1.000000,0.743145,0.669131,-5.000000e-01,-8.660254e-01,summer,-0.043022,-0.999074
224645,Canada,Stickers for Less,Holographic Goose,2016,11,1,306,1,0.781831,0.623490,0.207912,0.978148,-5.000000e-01,8.660254e-01,autumn,-0.849817,0.527078
79537,Norway,Stickers for Less,Kaggle Tiers,2012,6,2,154,5,-0.974928,-0.222521,0.406737,0.913545,1.224647e-16,-1.000000e+00,summer,0.471160,-0.882048
31844,Norway,Premium Sticker Mart,Kerneler Dark Mode,2010,12,20,354,0,0.000000,1.000000,-0.866025,-0.500000,-2.449294e-16,1.000000e+00,winter,-0.188227,0.982126
222978,Kenya,Discount Stickers,Kerneler,2016,10,13,287,3,0.433884,-0.900969,0.406737,-0.913545,-8.660254e-01,5.000000e-01,autumn,-0.974100,0.226116
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79669,Finland,Discount Stickers,Kerneler Dark Mode,2012,6,4,156,0,0.000000,1.000000,0.743145,0.669131,1.224647e-16,-1.000000e+00,summer,0.440519,-0.897743
192747,Kenya,Premium Sticker Mart,Kaggle Tiers,2015,11,12,316,3,0.433884,-0.900969,0.587785,-0.809017,-5.000000e-01,8.660254e-01,autumn,-0.746972,0.664855
112732,Kenya,Stickers for Less,Kaggle Tiers,2013,6,6,157,3,0.433884,-0.900969,0.951057,0.309017,1.224647e-16,-1.000000e+00,summer,0.425000,-0.905193
38733,Italy,Discount Stickers,Kerneler,2011,3,7,66,0,0.000000,1.000000,0.994522,0.104528,1.000000e+00,6.123234e-17,spring,0.907014,0.421101


In [36]:
# model.fit(X_train_first, target_first)

In [37]:
# baseline = model.predict(X_train_second)

In [38]:
# X_train_first['day'] = X_train_first['day'].astype('category')
# X_train_first['month'] = X_train_first['month'].astype('category')
# X_train_first['day of year'] = X_train_first['day of year'].astype('category')
# X_train_first['day of week'] = X_train_first['day of week'].astype('category')
# X_train_first['season'] = X_train_first['season'].astype('category')
# X_train_first['country'] = X_train_first['country'].astype('category')
# X_train_first['store'] = X_train_first['store'].astype('category')
# X_train_first['product'] = X_train_first['product'].astype('category')

# X_train_second['day'] = X_train_second['day'].astype('category')
# X_train_second['month'] = X_train_second['month'].astype('category')
# X_train_second['day of year'] = X_train_second['day of year'].astype('category')
# X_train_second['day of week'] = X_train_second['day of week'].astype('category')
# X_train_second['season'] = X_train_second['season'].astype('category')
# X_train_second['country'] = X_train_second['country'].astype('category')
# X_train_second['store'] = X_train_second['store'].astype('category')
# X_train_second['product'] = X_train_second['product'].astype('category')


In [39]:
# X_train_first['day']

In [40]:
# import xgboost as xgb

# xgb.set_config(verbosity=3)
# dtrain = xgb.DMatrix(X_train_first, target_first, enable_categorical=True)
# dtest = xgb.DMatrix(X_train_second, enable_categorical=True)

# params = { 'objective': 'reg:absoluteerror',  'max_depth': 16 }

# bst = xgb.train(params, dtrain, num_boost_round=500)

In [41]:
# baseline = bst.predict(dtest)

In [42]:
# pool_w_baseline = Pool(X_train_second, target_second, cat_features=['country', 'store', 'product', 'day', 'month', 'day of year', 'day of week', 'season'], baseline=baseline) # [np.mean(target)]*len(target)
from catboost import CatBoostRegressor, Pool

pool = Pool(df_train, target, cat_features=['country', 'store', 'product', 'day', 'month', 'day of year', 'day of week', 'season']) # [np.mean(target)]*len(target)

new_regressor = CatBoostRegressor(iterations=1000, loss_function='MAPE', max_depth=6, verbose=True)

In [43]:
new_regressor.fit(pool)

0:	learn: 0.8809941	total: 161ms	remaining: 2m 40s
1:	learn: 0.8578267	total: 263ms	remaining: 2m 11s
2:	learn: 0.8360609	total: 349ms	remaining: 1m 55s
3:	learn: 0.8150218	total: 441ms	remaining: 1m 49s
4:	learn: 0.7947201	total: 555ms	remaining: 1m 50s
5:	learn: 0.7744887	total: 627ms	remaining: 1m 43s
6:	learn: 0.7557004	total: 705ms	remaining: 1m 39s
7:	learn: 0.7375909	total: 785ms	remaining: 1m 37s
8:	learn: 0.7192808	total: 879ms	remaining: 1m 36s
9:	learn: 0.7016683	total: 964ms	remaining: 1m 35s
10:	learn: 0.6848106	total: 1.04s	remaining: 1m 33s
11:	learn: 0.6685497	total: 1.13s	remaining: 1m 32s
12:	learn: 0.6527860	total: 1.25s	remaining: 1m 35s
13:	learn: 0.6378676	total: 1.35s	remaining: 1m 35s
14:	learn: 0.6234242	total: 1.42s	remaining: 1m 33s
15:	learn: 0.6095965	total: 1.5s	remaining: 1m 32s
16:	learn: 0.5962280	total: 1.59s	remaining: 1m 31s
17:	learn: 0.5840939	total: 1.7s	remaining: 1m 32s
18:	learn: 0.5726575	total: 1.78s	remaining: 1m 31s
19:	learn: 0.5621099	tot

116:	learn: 0.3683785	total: 10.9s	remaining: 1m 22s
117:	learn: 0.3679817	total: 11s	remaining: 1m 22s
118:	learn: 0.3676082	total: 11.1s	remaining: 1m 22s
119:	learn: 0.3672483	total: 11.2s	remaining: 1m 22s
120:	learn: 0.3669084	total: 11.3s	remaining: 1m 21s
121:	learn: 0.3663428	total: 11.4s	remaining: 1m 22s
122:	learn: 0.3660104	total: 11.5s	remaining: 1m 22s
123:	learn: 0.3656894	total: 11.6s	remaining: 1m 22s
124:	learn: 0.3650398	total: 11.7s	remaining: 1m 21s
125:	learn: 0.3646923	total: 11.8s	remaining: 1m 22s
126:	learn: 0.3641548	total: 11.9s	remaining: 1m 22s
127:	learn: 0.3638600	total: 12.1s	remaining: 1m 22s
128:	learn: 0.3633612	total: 12.2s	remaining: 1m 22s
129:	learn: 0.3629757	total: 12.3s	remaining: 1m 22s
130:	learn: 0.3625070	total: 12.4s	remaining: 1m 22s
131:	learn: 0.3620904	total: 12.6s	remaining: 1m 22s
132:	learn: 0.3617553	total: 12.6s	remaining: 1m 22s
133:	learn: 0.3614774	total: 12.8s	remaining: 1m 22s
134:	learn: 0.3611073	total: 12.9s	remaining: 1m

In [44]:
df_test = pd.read_csv('test.csv')
ids = df_test['id']
df_test.drop('id', axis=1, inplace=True)
df_test

,date,country,store,product
0,2017-01-01,Canada,Discount Stickers,Holographic Goose
1,2017-01-01,Canada,Discount Stickers,Kaggle
2,2017-01-01,Canada,Discount Stickers,Kaggle Tiers
3,2017-01-01,Canada,Discount Stickers,Kerneler
4,2017-01-01,Canada,Discount Stickers,Kerneler Dark Mode
...,...,...,...,...
98545,2019-12-31,Singapore,Premium Sticker Mart,Holographic Goose
98546,2019-12-31,Singapore,Premium Sticker Mart,Kaggle
98547,2019-12-31,Singapore,Premium Sticker Mart,Kaggle Tiers
98548,2019-12-31,Singapore,Premium Sticker Mart,Kerneler


In [45]:
df_test['year'] = df_test['date'].apply(lambda x: int(x[:4]))
df_test['month'] = df_test['date'].apply(lambda x: int(x[5:7]))
df_test['day'] = df_test['date'].apply(lambda x: int(x[8:10]))
df_test['date'] = pd.to_datetime(df_test['date'])
df_test['day of year'] = df_test['date'].dt.dayofyear
df_test['day of week'] = df_test['date'].dt.weekday
df_test['sin day of week'] = np.sin(2 * np.pi * df_test['day of week'] / 7)
df_test['cos day of week'] = np.cos(2 * np.pi * df_test['day of week'] / 7)
df_test['sin day of month'] = np.sin(2 * np.pi * df_test['day'] / 30)
df_test['cos day of month'] = np.cos(2 * np.pi * df_test['day'] / 30)
df_test['sin of month'] = np.sin(2 * np.pi * df_test['month'] / 12)
df_test['cos of month'] = np.cos(2 * np.pi * df_test['month'] / 12)
#df_test['seconds since 1970'] = df_test['date'].astype('int64') // 10 ** 9
df_test.drop('date', axis=1, inplace=True) #
df_test

,country,store,product,year,month,day,day of year,day of week,sin day of week,cos day of week,sin day of month,cos day of month,sin of month,cos of month
0,Canada,Discount Stickers,Holographic Goose,2017,1,1,1,6,-0.781831,0.62349,0.207912,0.978148,5.000000e-01,0.866025
1,Canada,Discount Stickers,Kaggle,2017,1,1,1,6,-0.781831,0.62349,0.207912,0.978148,5.000000e-01,0.866025
2,Canada,Discount Stickers,Kaggle Tiers,2017,1,1,1,6,-0.781831,0.62349,0.207912,0.978148,5.000000e-01,0.866025
3,Canada,Discount Stickers,Kerneler,2017,1,1,1,6,-0.781831,0.62349,0.207912,0.978148,5.000000e-01,0.866025
4,Canada,Discount Stickers,Kerneler Dark Mode,2017,1,1,1,6,-0.781831,0.62349,0.207912,0.978148,5.000000e-01,0.866025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98545,Singapore,Premium Sticker Mart,Holographic Goose,2019,12,31,365,1,0.781831,0.62349,0.207912,0.978148,-2.449294e-16,1.000000
98546,Singapore,Premium Sticker Mart,Kaggle,2019,12,31,365,1,0.781831,0.62349,0.207912,0.978148,-2.449294e-16,1.000000
98547,Singapore,Premium Sticker Mart,Kaggle Tiers,2019,12,31,365,1,0.781831,0.62349,0.207912,0.978148,-2.449294e-16,1.000000
98548,Singapore,Premium Sticker Mart,Kerneler,2019,12,31,365,1,0.781831,0.62349,0.207912,0.978148,-2.449294e-16,1.000000


In [46]:
df_test['season'] = df_test['month'].apply(season)

In [47]:
df_test['day'] = df_test['day'].astype('category')
df_test['month'] = df_test['month'].astype('category')

df_test['sin day of year'] = np.sin(2 * np.pi * df_test['day of year'] / 365)
df_test['cos day of year'] = np.cos(2 * np.pi * df_test['day of year'] / 365)

df_test['day of year'] = df_test['day of year'].astype('category')
df_test['day of week'] = df_test['day of week'].astype('category')


df_test

,country,store,product,year,month,day,day of year,day of week,sin day of week,cos day of week,sin day of month,cos day of month,sin of month,cos of month,season,sin day of year,cos day of year
0,Canada,Discount Stickers,Holographic Goose,2017,1,1,1,6,-0.781831,0.62349,0.207912,0.978148,5.000000e-01,0.866025,winter,1.721336e-02,0.999852
1,Canada,Discount Stickers,Kaggle,2017,1,1,1,6,-0.781831,0.62349,0.207912,0.978148,5.000000e-01,0.866025,winter,1.721336e-02,0.999852
2,Canada,Discount Stickers,Kaggle Tiers,2017,1,1,1,6,-0.781831,0.62349,0.207912,0.978148,5.000000e-01,0.866025,winter,1.721336e-02,0.999852
3,Canada,Discount Stickers,Kerneler,2017,1,1,1,6,-0.781831,0.62349,0.207912,0.978148,5.000000e-01,0.866025,winter,1.721336e-02,0.999852
4,Canada,Discount Stickers,Kerneler Dark Mode,2017,1,1,1,6,-0.781831,0.62349,0.207912,0.978148,5.000000e-01,0.866025,winter,1.721336e-02,0.999852
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98545,Singapore,Premium Sticker Mart,Holographic Goose,2019,12,31,365,1,0.781831,0.62349,0.207912,0.978148,-2.449294e-16,1.000000,winter,6.432491e-16,1.000000
98546,Singapore,Premium Sticker Mart,Kaggle,2019,12,31,365,1,0.781831,0.62349,0.207912,0.978148,-2.449294e-16,1.000000,winter,6.432491e-16,1.000000
98547,Singapore,Premium Sticker Mart,Kaggle Tiers,2019,12,31,365,1,0.781831,0.62349,0.207912,0.978148,-2.449294e-16,1.000000,winter,6.432491e-16,1.000000
98548,Singapore,Premium Sticker Mart,Kerneler,2019,12,31,365,1,0.781831,0.62349,0.207912,0.978148,-2.449294e-16,1.000000,winter,6.432491e-16,1.000000


In [48]:
# df_test['country'] = df_test['country'].astype('category')
# df_test['store'] = df_test['store'].astype('category')
# df_test['product'] = df_test['product'].astype('category')
# df_test['season'] = df_test['season'].astype('category')

In [49]:
# dtest = xgb.DMatrix(df_test, enable_categorical=True)
# a = bst.predict(dtest)

In [51]:
pred = new_regressor.predict(df_test) #+ model.predict(df_test)# [:, 0] # np.array([np.mean(target)] * len(df_test))  
pred

array([ 283.3636058 ,  839.15908766,  655.11700925, ..., 1292.69616164,
        801.59119449,  902.82177645])

In [52]:
df_pred = pd.DataFrame(pred, columns=['num_sold'])
df_pred['id'] = ids
df_pred = df_pred[['id', 'num_sold']]
df_pred

,id,num_sold
0,230130,283.363606
1,230131,839.159088
2,230132,655.117009
3,230133,368.960099
4,230134,449.954719
...,...,...
98545,328675,292.004517
98546,328676,1644.820021
98547,328677,1292.696162
98548,328678,801.591194


In [53]:
df_pred.to_csv('submission.csv', index=False)